In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
query="Which is the most successful team in the history of the IPL?"

In [3]:
import requests

response = requests.get(
    "http://localhost:8888/search",
    params={
        "q": query,
        "format": "json",

        # categories
        "categories": "general",

        # language
        "language": "en-IN",

        # time range
        "time_range": "month",  # day, month, year

        # safe search
        "safesearch": 0,   # 0=None, 1=Moderate, 2=Strict

        # pagination
        "pageno": 1,

    }
)

data = response.json()

print(data.keys())

dict_keys(['query', 'results', 'answers', 'corrections', 'infoboxes', 'suggestions', 'unresponsive_engines'])


In [4]:
for item in data["results"]:
    print(f"Title: {item['title']}")
    print(f"URL: {item['url']}")
    print("-" * 80)

urls = []
for item in data["results"]:
    urls.append(item["url"])
urls = urls[:7]

Title: List of Indian Premier League records and statistics - Wikipedia
URL: https://en.wikipedia.org/wiki/List_of_Indian_Premier_League_records_and_statistics
--------------------------------------------------------------------------------
Title: Check out the most successful IPL captains in history, but do you ...
URL: https://www.instagram.com/reel/DYpIMWCCDXZ/
--------------------------------------------------------------------------------
Title: Indian Premier League Records | All-Time Stats, Players ... - Britannica
URL: https://www.britannica.com/sports/Indian-Premier-League-Records
--------------------------------------------------------------------------------
Title: Most Winning Team in IPL | Complete Record List & Stats
URL: https://www.sportsdunia.com/ipl-records/most-winning-team-ipl
--------------------------------------------------------------------------------
Title: IPL Teams Finals & Trophies IPL teams with most finals ... - Facebook
URL: https://www.facebook.com/6155

#### Fetching WebPage

In [5]:
import trafilatura

url = "https://www.google.com"
downloaded = trafilatura.fetch_url(url)
result = trafilatura.extract(downloaded, include_comments=False, include_tables=False) 
print(result)

Gmail
Images
Sign in
Advanced search
Google offered in:
हिन्दी
বাংলা
తెలుగు
मराठी
தமிழ்
ગુજરાતી
ಕನ್ನಡ
മലയാളം
ਪੰਜਾਬੀ
Advertising
Business Solutions
About Google
Google.co.in
© 2026 -
Privacy
-
Terms
Google apps


In [6]:
import trafilatura
from concurrent.futures import ThreadPoolExecutor, as_completed

def scrape_url(url):
    try:
        downloaded = trafilatura.fetch_url(url)
        result = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=False
        )
        return url, result
    except Exception as e:
        return url, f"Error: {e}"

content = {}

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(scrape_url, url) for url in urls]

    for future in as_completed(futures):
        url, result = future.result()
        content[url] = result

print(len(content))
print(content.keys())

7
dict_keys(['https://testbook.com/news/ipl-winners-list-all-season-notes-pdf/', 'https://en.wikipedia.org/wiki/List_of_Indian_Premier_League_records_and_statistics', 'https://www.facebook.com/61556908740578/posts/ipl-teams-finals-trophies-ipl-teams-with-most-finals-appearances-and-trophiescsk/122311694504230291/', 'https://www.britannica.com/sports/Indian-Premier-League-Records', 'https://thecricscope.com/cricket-statistics/ipl-winners-list-all-seasons/', 'https://www.instagram.com/reel/DYpIMWCCDXZ/', 'https://www.sportsdunia.com/ipl-records/most-winning-team-ipl'])


#### Chunking the Documents

In [7]:
import hashlib

def chunk_text(text, chunk_size=400, overlap=50):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

def build_chunks(content: dict) -> list:
    all_chunks = []

    for url, text in content.items():
        if not text or text.startswith("Error:"):
            continue

        domain = url.split("//")[-1].split("/")[0].replace("www.", "")
        raw_chunks = chunk_text(text)

        for idx, chunk_text_str in enumerate(raw_chunks):
            chunk_id = hashlib.md5(f"{url}-{idx}".encode()).hexdigest()[:12]
            all_chunks.append({
                "chunk_id": chunk_id,
                "url": url,
                "domain": domain,
                "text": chunk_text_str,
                "chunk_index": idx,
                "total_chunks": len(raw_chunks),
                "metadata": {
                    "char_count": len(chunk_text_str),
                    "word_count": len(chunk_text_str.split()),
                }
            })

    return all_chunks

chunks = build_chunks(content)
chunks

[{'chunk_id': '1dbcdda9cf78',
  'url': 'https://testbook.com/news/ipl-winners-list-all-season-notes-pdf/',
  'domain': 'testbook.com',
  'text': 'The Indian Premier League is a professional T20 cricket league in India. It is organized every year by the BCCI since 2008. It features top international and domestic players competing across city based franchises. Over the years, IPL has become the most popular T20 league in the world. It is known for its high quality matches, global viewership and competitive format. IPL is an important topic in Sports Current Affairs for General Awareness section of government exams. Many questions are asked about IPL winners list, Orange Cap, Purple Cap, and recent champions in competitive exams. IPL Winners List & IPL Trophy Winners List 2008 to 2025 The IPL winners list from 2008 to 2025 shows the dominance of teams like Mumbai Indians and Chennai Super Kings. In 2025, Royal Challengers Bangalore won their maiden title. IPL Trophy Winners & Runner Up Li

#### Reranking Chunks

In [ ]:
# from huggingface_hub import InferenceClient
# import numpy as np


# # Load client once
# client = InferenceClient(api_key=os.getenv("HUGGINGFACE_API_KEY"))

# BI_MODEL = "BAAI/bge-base-en-v1.5"
# CE_MODEL = "  "

# # --- Stage 1: Bi-Encoder (coarse retrieval) ---
# def bi_encode_chunks(chunks: list) -> np.ndarray:
#     texts = [c["text"] for c in chunks]
#     embeddings = [
#         client.feature_extraction(text, model=BI_MODEL)
#         for text in texts
#     ]
#     emb_array = np.array(embeddings)
#     # normalize
#     norms = np.linalg.norm(emb_array, axis=1, keepdims=True)
#     return emb_array / norms

# def retrieve_top_k(query: str, chunks: list, chunk_embeddings: np.ndarray, k=50) -> list:
#     query_emb = np.array(client.feature_extraction(query, model=BI_MODEL))
#     query_emb = query_emb / np.linalg.norm(query_emb)
#     scores = np.dot(chunk_embeddings, query_emb).squeeze()
#     top_k_idx = np.argsort(scores)[::-1][:k]
#     return [(chunks[i], float(scores[i])) for i in top_k_idx]

# # --- Stage 2: Cross-Encoder (fine reranking) ---
# def _truncate_for_rerank(text: str, max_words=200) -> str:
#     words = text.split()
#     if len(words) <= max_words:
#         return text
#     return " ".join(words[:max_words])

# def rerank(query: str, candidates: list, top_n=5) -> list:
#     ce_scores = [
#         client.text_classification(
#             f"{query} [SEP] {_truncate_for_rerank(c['text'])}",
#             model=CE_MODEL
#         )[0].score
#         for c, _ in candidates
#     ]

#     ranked = sorted(
#         zip(candidates, ce_scores),
#         key=lambda x: x[1],
#         reverse=True
#     )

#     return [
#         {**chunk, "bi_score": bi_score, "ce_score": float(ce_score)}
#         for (chunk, bi_score), ce_score in ranked[:top_n]
#     ]

In [9]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

flag = True
# Load models once
if flag:
    bi_encoder = SentenceTransformer("BAAI/bge-base-en-v1.5")
    cross_encoder = CrossEncoder("BAAI/bge-reranker-base")

/Users/hariom/Perplexity Clone/backend/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10374.66it/s]


In [10]:
# --- Stage 1: Bi-Encoder (coarse retrieval) ---
def bi_encode_chunks(chunks: list) -> np.ndarray:
    texts = [c["text"] for c in chunks]
    return bi_encoder.encode(texts, normalize_embeddings=True, show_progress_bar=False)

def retrieve_top_k(query: str, chunks: list, chunk_embeddings: np.ndarray, k=50) -> list:
    query_emb = bi_encoder.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embeddings, query_emb.T).squeeze()
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top_k_idx]

# --- Stage 2: Cross-Encoder (fine reranking) ---
def rerank(query: str, candidates: list, top_n=5) -> list:
    # candidates = [(chunk_dict, bi_score), ...]
    pairs = [[query, c["text"]] for c, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)
    
    ranked = sorted(
        zip(candidates, ce_scores),
        key=lambda x: x[1],
        reverse=True
    )
    
    return [
        {**chunk, "bi_score": bi_score, "ce_score": float(ce_score)}
        for (chunk, bi_score), ce_score in ranked[:top_n]
    ]


#### Pipeline

In [11]:
def search(query: str, chunks: list, chunk_embeddings: np.ndarray, k=50, top_n=5):
    candidates = retrieve_top_k(query, chunks, chunk_embeddings, k=k)
    results = rerank(query, candidates, top_n=top_n)
    return results

chunk_embeddings = bi_encode_chunks(chunks)
results = search(query, chunks, chunk_embeddings, k=50, top_n=5)
# print(len(results))
print(results)

[{'chunk_id': '8726412e01e6', 'url': 'https://www.britannica.com/sports/Indian-Premier-League-Records', 'domain': 'britannica.com', 'text': '2026 - Abhishek Sharma (SRH): 141 runs off 55 balls against PBKS in 2025 - Quinton de Kock (LSG): 140* runs off 70 balls against KKR in 2022 - Abhishek Sharma (SRH): 135* runs off 68 balls against DC in 2026 - A.B. de Villiers (RCB): 133* runs off 59 balls against MI in 2015 *Batter was not dismissed. Fastest Centurions- Chris Gayle (RCB): 30 balls against PWI in 2013 - Vaibhav Suryavanshi (RR): 35 balls against GT in 2025 - Vaibhav Suryavanshi (RR): 36 balls against SRH in 2026 - Yusuf Pathan (RR): 37 balls against MI in 2010 - Heinrich Klaasen (SRH): 37 balls against KKR in 2025 - David Miller (PBKS): 38 balls against RCB in 2013 - Virat Kohli (RCB): 8 - Jos Buttler (GT/MI/RR): 7 - Chris Gayle (KKR/PBKS/RCB): 6 - K.L. Rahul (DC/LSG/PBKS/RCB/SRH): 6 - Sanju Samson (CSK/DC/RR): 5 - Shubman Gill (GT/KKR): 4 - Shane Watson (CSK/RCB/RR): 4 - David Wa

In [12]:
# Keep the same index for identical URLs
url_to_idx = {}
context_lines = []

for r in results:
    url = r["url"]
    if url not in url_to_idx:
        url_to_idx[url] = len(url_to_idx) + 1
    idx = url_to_idx[url]
    context_lines.append(f"[{idx}] {url}\n{r['text']}")

context = "\n\n".join(context_lines)
print(context)

[1] https://www.britannica.com/sports/Indian-Premier-League-Records
2026 - Abhishek Sharma (SRH): 141 runs off 55 balls against PBKS in 2025 - Quinton de Kock (LSG): 140* runs off 70 balls against KKR in 2022 - Abhishek Sharma (SRH): 135* runs off 68 balls against DC in 2026 - A.B. de Villiers (RCB): 133* runs off 59 balls against MI in 2015 *Batter was not dismissed. Fastest Centurions- Chris Gayle (RCB): 30 balls against PWI in 2013 - Vaibhav Suryavanshi (RR): 35 balls against GT in 2025 - Vaibhav Suryavanshi (RR): 36 balls against SRH in 2026 - Yusuf Pathan (RR): 37 balls against MI in 2010 - Heinrich Klaasen (SRH): 37 balls against KKR in 2025 - David Miller (PBKS): 38 balls against RCB in 2013 - Virat Kohli (RCB): 8 - Jos Buttler (GT/MI/RR): 7 - Chris Gayle (KKR/PBKS/RCB): 6 - K.L. Rahul (DC/LSG/PBKS/RCB/SRH): 6 - Sanju Samson (CSK/DC/RR): 5 - Shubman Gill (GT/KKR): 4 - Shane Watson (CSK/RCB/RR): 4 - David Warner (DC/SRH): 4 - 973, Virat Kohli (RCB; 2016) - 890, Shubman Gill (GT; 

#### LLM Synthesis

In [ ]:
SYSTEM_PROMPT = """
You are an advanced AI answer engine optimized for concise, accurate, citation-based responses.

Your job:
- Answer using ONLY the provided context.
- Synthesize information instead of copying raw text.
- Keep answers concise but information-dense.
- Always include citations from the provided sources.
- Never hallucinate facts.
- If the context is insufficient, explicitly say:
  "I could not find enough reliable information in the provided context."

RESPONSE RULES:
1. Start directly with the answer.
2. Do NOT mention "based on the context".
3. Do NOT dump raw search results.
4. Combine duplicate information from multiple sources.
5. Prefer the most relevant and trustworthy sources.
6. Use short paragraphs or bullet points.
7. Keep a professional Perplexity-style tone.
8. If multiple sources support a claim, cite all of them.

CITATION FORMAT:
- Use inline markdown citations.
- Format:
  [Source 1]
  [Source 1][Source 2]

EXAMPLE:
MS Dhoni won the most trophies as a captain. [1][3]

SOURCE HANDLING:
- Each context item may contain:
  - url
  - snippet/content
  

IMPORTANT:
- Ignore irrelevant context.
- Remove duplicated lines/content.
- Never expose internal reasoning.
- Never say "according to the retrieved context".
- Never fabricate citations.
"""

message = f"""
User Question:
{query}

Retrieved Context:
{context}

Generate a concise, high-quality answer with inline citations.
"""

In [14]:
# SYSTEM_PROMPT = """You are a helpful assistant that provides concise and accurate answers based on the provided context. 
# Use the following retrieved information to answer the user's question. If the information is insufficient, say you don't know.
# """

# message = f"""query: {query}

# context: {results}"""

In [17]:
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage

model = init_chat_model("gemma-4-31b-it", model_provider="google_genai", api_key=os.getenv("GEMINI_API_KEY"))

messages = [
    SystemMessage(SYSTEM_PROMPT),
    HumanMessage(message)
]
response = model.invoke(messages)
print(response.content[-1]["text"])

The **Mumbai Indians (MI)** and **Chennai Super Kings (CSK)** are the most successful teams in IPL history, having each won five titles [1][2]. 

Their specific achievements include:
*   **Titles:** MI won in 2013, 2015, 2017, 2019, and 2020; CSK won in 2010, 2011, 2018, 2021, and 2023 [1].
*   **Match Records:** The Mumbai Indians hold the record for both the most matches played and the most matches won [2].
*   **Win Percentage:** While MI and CSK have the most titles, the Gujarat Titans (GT) hold the highest win percentage at 62.22% [2].
